In [1]:
"""
Actions that we need to train the adapter on:
- LGTM - Accept the draft as-is
- add content
- add tool call at a specific index
- replace content
- replace tool call 
- delete tool call at a specific index
- delete content

How should we modify the data so that the adapter can perform the above actions?
- LGTM: keep data as-is
- add content: delete content when gold content is present.
- add tool call: drop tool call at random indices
- replace content: corrupt the content.
- replace tool call:
    - corrupt the tool call name at index I.
    - corrupt the tool call arguments at index I.
- delete tool call: add random tool call at random indices.
- delete content: add content to when no gold content is present.

Sometimes the adapter model can take multiple actions in a single turn. Some actions cannot be performed together.
- no other action allowed if LGTM is performed.
- you can either add content or replace content or delete content but not do them together.
- you cannot add a tool call at index I and delete a tool call at the same index I. Should do replace tool call at index I.
"""

'\nActions that we need to train the adapter on:\n- LGTM - Accept the draft as-is\n- add content\n- add tool call at a specific index\n- replace content\n- replace tool call \n- delete tool call at a specific index\n- delete content\n\nHow should we modify the data so that the adapter can perform the above actions?\n- LGTM: keep data as-is\n- add content: delete content when gold content is present.\n- add tool call: drop tool call at random indices\n- replace content: corrupt the content.\n- replace tool call:\n    - corrupt the tool call name at index I.\n    - corrupt the tool call arguments at index I.\n- delete tool call: add random tool call at random indices.\n- delete content: add content to when no gold content is present.\n\nSometimes the adapter model can take multiple actions in a single turn. Some actions cannot be performed together.\n- no other action allowed if LGTM is performed.\n- you can either add content or replace content or delete content but not do them together

In [2]:
# building training data for the adapter

import json

hermes_data_path = './data/raw/hermes_func_calling.jsonl'
hermes_data = []
with open(hermes_data_path, 'r') as f:
    for line in f:
        hermes_data.append(json.loads(line))

hermes_data[0]



{'source_dataset': 'NousResearch/hermes-function-calling-v1',
 'source_config': 'func_calling',
 'source_row_idx': 0,
 'id': '82088305-310b-45cb-ac76-ab273503b5cd',
 'conversations': [{'from': 'system',
   'value': 'You are a function calling AI model. You are provided with function signatures within <tools> </tools> XML tags. You may call one or more functions to assist with the user query. Don\'t make assumptions about what values to plug into functions.\n<tools>\n[{"type": "function", "function": {"name": "get_camera_live_feed", "description": "Retrieves the live feed from a specified security camera.", "parameters": {"type": "object", "properties": {"camera_id": {"type": "string", "description": "The unique identifier for the camera."}, "stream_quality": {"type": "string", "description": "The desired quality of the live stream.", "enum": ["720p", "1080p", "4k"]}}, "required": ["camera_id"]}}}, {"type": "function", "function": {"name": "list_all_cameras", "description": "Lists all t

In [3]:
# convert to messages format
from typing import Any

ROLE_MAP = {
    "system": "system",
    "human": "user",
    "gpt": "assistant",
    "tool": "tool",
}

def _coerce_message(turn: dict[str, Any]) -> dict[str, str] | None:
    from_role = turn.get("from")
    value = turn.get("value")
    if not isinstance(from_role, str) or not isinstance(value, str):
        return None

    role = ROLE_MAP.get(from_role)
    if role is None:
        return None
    return {"role": role, "content": value}

_coerce_message(hermes_data[0]['conversations'][0])

{'role': 'system',
 'content': 'You are a function calling AI model. You are provided with function signatures within <tools> </tools> XML tags. You may call one or more functions to assist with the user query. Don\'t make assumptions about what values to plug into functions.\n<tools>\n[{"type": "function", "function": {"name": "get_camera_live_feed", "description": "Retrieves the live feed from a specified security camera.", "parameters": {"type": "object", "properties": {"camera_id": {"type": "string", "description": "The unique identifier for the camera."}, "stream_quality": {"type": "string", "description": "The desired quality of the live stream.", "enum": ["720p", "1080p", "4k"]}}, "required": ["camera_id"]}}}, {"type": "function", "function": {"name": "list_all_cameras", "description": "Lists all the security cameras connected to the home network.", "parameters": {"type": "object", "properties": {"include_offline": {"type": "boolean", "description": "Whether to include cameras t

In [4]:
for example in hermes_data:
    messages = []
    for turn in example['conversations']:
        message = _coerce_message(turn)
        if message is not None:
            messages.append(message)
    example['messages'] = messages
hermes_data[0]

{'source_dataset': 'NousResearch/hermes-function-calling-v1',
 'source_config': 'func_calling',
 'source_row_idx': 0,
 'id': '82088305-310b-45cb-ac76-ab273503b5cd',
 'conversations': [{'from': 'system',
   'value': 'You are a function calling AI model. You are provided with function signatures within <tools> </tools> XML tags. You may call one or more functions to assist with the user query. Don\'t make assumptions about what values to plug into functions.\n<tools>\n[{"type": "function", "function": {"name": "get_camera_live_feed", "description": "Retrieves the live feed from a specified security camera.", "parameters": {"type": "object", "properties": {"camera_id": {"type": "string", "description": "The unique identifier for the camera."}, "stream_quality": {"type": "string", "description": "The desired quality of the live stream.", "enum": ["720p", "1080p", "4k"]}}, "required": ["camera_id"]}}}, {"type": "function", "function": {"name": "list_all_cameras", "description": "Lists all t

In [5]:
# check if all system prompt start with "You are a function calling AI model"
for example in hermes_data:
    if example['messages'][0]['role'] == 'system' and not example['messages'][0]['content'].startswith('You are a function calling AI model'):
        print(example['messages'][0]['content'])

You are an expert structured information extraction AI model. You will be provided with documents to extract information from. You are also provided with the json schema to output extracted information in the function signatures within XML tags <tools></tools>. Don't make assumptions about what values to plug into json schema. 
<tools>
[{"type": "function", "function": {"name": "ExpertQAExtractor", "description": "Extracts a list of open-ended questions related to the document, that are potentially ambiguous.", "parameters": {"type": "object", "properties": {"open_ended_questions": {"type": "array", "items": {"type": "string"}}}, "required": ["open_ended_questions"]}}}]
</tools>
For each extraction function call return a json object with function name and arguments followed by a <tool_call> tag with the following schema:
<tool_call>
{"name": <function-name>, "arguments": <args-dict>}
</tool_call>
You are an expert structured information extraction AI model. You will be provided with do

In [6]:
# check if all system prompt end with 
# <tool_call>
# {"name": <function-name>, "arguments": <args-dict>}
# </tool_call>

s = """
<tool_call>
{"name": <function-name>, "arguments": <args-dict>}
</tool_call>
""".strip()

for example in hermes_data:
    if example['messages'][0]['role'] == 'system' and not example['messages'][0]['content'].strip().endswith(s):
        print(example['messages'][0]['content'])

In [7]:
# Unique ways to telling the model to output a tool call
unique_tool_call_prompts = set()
for example in hermes_data:
    if example['messages'][0]['role'] == 'system':
        prompt = example['messages'][0]['content'].strip().splitlines()[-4]
        unique_tool_call_prompts.add(prompt)

len(unique_tool_call_prompts)

2

In [8]:
# unique system prompts
unique_system_prompts = set()
for example in hermes_data:
    if example['messages'][0]['role'] == 'system':
        unique_system_prompts.add(example['messages'][0]['content'].strip())

len(unique_system_prompts)

1113

In [9]:
len(hermes_data)

1893

In [10]:
for x in unique_system_prompts:
    print(x)
    break



You are a function calling AI model. You are provided with function signatures within <tools> </tools> XML tags. You may call one or more functions to assist with the user query. Don't make assumptions about what values to plug into functions.
<tools>
[{"type": "function", "function": {"name": "create_ad_campaign", "description": "Create a new advertising campaign with specified parameters on multiple platforms.", "parameters": {"type": "object", "properties": {"campaign_name": {"type": "string", "description": "The name of the advertising campaign."}, "platforms": {"type": "array", "description": "List of platforms where the campaign will run.", "items": {"type": "string"}}, "budget": {"type": "number", "description": "Total budget allocated for the campaign."}, "start_date": {"type": "string", "description": "The start date of the campaign in YYYY-MM-DD format."}, "end_date": {"type": "string", "description": "The end date of the campaign in YYYY-MM-DD format."}, "target_audience": {

In [11]:
print(next(iter(unique_system_prompts)))

You are a function calling AI model. You are provided with function signatures within <tools> </tools> XML tags. You may call one or more functions to assist with the user query. Don't make assumptions about what values to plug into functions.
<tools>
[{"type": "function", "function": {"name": "create_ad_campaign", "description": "Create a new advertising campaign with specified parameters on multiple platforms.", "parameters": {"type": "object", "properties": {"campaign_name": {"type": "string", "description": "The name of the advertising campaign."}, "platforms": {"type": "array", "description": "List of platforms where the campaign will run.", "items": {"type": "string"}}, "budget": {"type": "number", "description": "Total budget allocated for the campaign."}, "start_date": {"type": "string", "description": "The start date of the campaign in YYYY-MM-DD format."}, "end_date": {"type": "string", "description": "The end date of the campaign in YYYY-MM-DD format."}, "target_audience": {

In [12]:
# drop tools
# drop tool call prompt

import re
ptrn = re.compile(r"<tools>\n.*\n<\/tools>", flags=re.DOTALL)

# This is a formatting error in the system prompt for hermes. we will drop these examples 
bad_s = """You are an expert structured information extraction AI model. You will be provided with documents to extract information from. You are also provided with the json schema to output extracted information in the function signatures within XML tags <tools></tools>. Don't make assumptions about what values to plug into json schema. {"type": "function", "function": {"name": "ExpertQAExtractor", "description": "Extracts a list of questions that ask how ideas in the document 
are connected or relate to each other. These identify relationships between concepts.", "parameters":"""


cleaned_examples = []
for example in hermes_data:
    system_prompt = example['messages'][0]['content']
    original_system_prompt = system_prompt

    # drop tool call prompt
    system_prompt = system_prompt.strip().splitlines()[:-4]
    system_prompt = "\n".join(system_prompt)

    # drop tool calls
    just_before_dropping_tool_calls = system_prompt
    system_prompt = ptrn.sub("", system_prompt).strip()
    if system_prompt.strip() == bad_s: continue

    example['messages'][0]['content'] = system_prompt
    cleaned_examples.append(example)
    
len(hermes_data), len(cleaned_examples)

(1893, 1832)

In [13]:
example['messages'][0]['content']

"You are an expert structured information extraction AI model. You will be provided with documents to extract information from. You are also provided with the json schema to output extracted information in the function signatures within XML tags <tools></tools>. Don't make assumptions about what values to plug into json schema."

In [73]:
ptrn

re.compile(r'<tools>\n.*\n<\/tools>', re.DOTALL|re.UNICODE)

In [14]:
# convert tool def into openai compatible tool def

tool_call_ptrn = re.compile(r"<tools>\n(.*)\n<\/tools>", flags=re.DOTALL)
for example in cleaned_examples:
    original_system_prompt = example['conversations'][0]['value']

    matches = re.search(tool_call_ptrn, original_system_prompt)
    if matches is None or len(matches.groups()) < 1: raise ValueError('matches is None')
    group = matches.groups()[0]
    group = group.strip()

    if not group: raise ValueError('group is empty')
    if '\n' in group: raise ValueError('group contains newlines. didn\'t expect that.')

    # tools are openai compatible tool defs
    tools = json.loads(group)
    example['tools'] = tools

In [121]:
tools

[{'type': 'function',
  'function': {'name': 'ExpertQAExtractor',
   'description': 'Extracts a list of questions that focus on summarizing a specific topic found in the document.',
   'parameters': {'type': 'object',
    'properties': {'topic_summarization_questions': {'type': 'array',
      'items': {'type': 'string'}}},
    'required': ['topic_summarization_questions']}}}]

In [86]:
example['messages'][1]['content']

'I\'ve recently installed a new security system at my home, and I want to ensure everything is functioning as it should. Specifically, I\'d like to start by checking the live feed from the camera located at the front door to monitor any activity. The camera has a unique identifier, which I\'ve already configured to be "front_door." I\'d prefer to view the live stream in high definition, so a 1080p quality would be ideal. Could you please call the appropriate function to retrieve the live feed from my front door camera in 1080p quality and provide me with the link to the stream?\n\nFollowing this, I would also like to record the live feed from this camera for the next 30 minutes. This is to test the recording feature and to keep an archived copy for security purposes. Please initiate the recording function for the "front_door" camera with a recording duration of 30 minutes.\n\nLastly, as part of my routine surveillance checks, I need to review footage from yesterday between 3 PM and 5 P

In [15]:
from dotenv import load_dotenv; load_dotenv()
from openai import OpenAI

client = OpenAI(base_url='http://localhost:8000/v1', api_key='dummy')
response = client.chat.completions.create(
    model='glm-4.7-fp8',
    messages=[
        {"role": "user", "content": example['messages'][1]['content']}
    ],
    tools=tools,
    tool_choice='auto',
)
print(response.choices[0].message)

ChatCompletionMessage(content="I'll help you extract topic-summarizing questions from this passage. This appears to be from a linear algebra textbook covering systems of linear equations, matrix notation, and vector definitions.", refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='chatcmpl-tool-81d6279eed31cd99', function=Function(arguments='{"topic_summarization_questions": ["What are the key definitions and notation systems introduced for representing matrices and column vectors in linear algebra?", "How does matrix notation simplify the representation and solution of systems of linear equations?", "What is the purpose and structure of the coefficient matrix, vector of constants, and solution vector in linear systems?", "How does the matrix representation LS(A, b) provide a compact notation for linear equation systems?", "What are the fundamental concepts and advantages of using matrix and vector not

In [16]:
response.choices[0].message.tool_calls

[ChatCompletionMessageFunctionToolCall(id='chatcmpl-tool-81d6279eed31cd99', function=Function(arguments='{"topic_summarization_questions": ["What are the key definitions and notation systems introduced for representing matrices and column vectors in linear algebra?", "How does matrix notation simplify the representation and solution of systems of linear equations?", "What is the purpose and structure of the coefficient matrix, vector of constants, and solution vector in linear systems?", "How does the matrix representation LS(A, b) provide a compact notation for linear equation systems?", "What are the fundamental concepts and advantages of using matrix and vector notation when working with linear systems?"]}', name='ExpertQAExtractor'), type='function')]

In [17]:
example = hermes_data[0]
for x in example['messages']:
    if x['role'] == 'assistant':
        print(x['content'])
        break





<tool_call>
{"name": "get_camera_live_feed", "arguments": {"camera_id": "front_door", "stream_quality": "1080p"}}
</tool_call>
<tool_call>
{"name": "record_camera_feed", "arguments": {"camera_id": "front_door", "duration": 30}}
</tool_call>
<tool_call>
{"name": "get_recorded_feed", "arguments": {"camera_id": "front_garden", "start_time": "2023-04-22T15:00:00Z", "end_time": "2023-04-22T17:00:00Z"}}
</tool_call>



In [18]:
from copy import deepcopy

separated_examples = []
for example in cleaned_examples:
    # count 'assistant' messages
    for i, x in enumerate(example['messages']):
        if x['role'] == 'assistant':
            _tmp = deepcopy(example)
            _tmp['messages'] = _tmp['messages'][:i+1]
            separated_examples.append(_tmp)

len(cleaned_examples), len(separated_examples)


(1832, 2759)

# Corrupt the messages

In [19]:
# convert to hf dataset obj
import pandas as pd

df = pd.DataFrame(separated_examples)
df.head()


,source_dataset,source_config,source_row_idx,id,conversations,tools,category,subcategory,task,messages
0,NousResearch/hermes-function-calling-v1,func_calling,0,82088305-310b-45cb-ac76-ab273503b5cd,"[{'from': 'system', 'value': 'You are a functi...","[{'type': 'function', 'function': {'name': 'ge...",IoT and Home Automation,Security Camera Management,View and Manage Security Camera Feeds,"[{'role': 'system', 'content': 'You are a func..."
1,NousResearch/hermes-function-calling-v1,func_calling,0,82088305-310b-45cb-ac76-ab273503b5cd,"[{'from': 'system', 'value': 'You are a functi...","[{'type': 'function', 'function': {'name': 'ge...",IoT and Home Automation,Security Camera Management,View and Manage Security Camera Feeds,"[{'role': 'system', 'content': 'You are a func..."
2,NousResearch/hermes-function-calling-v1,func_calling,1,bd48c5c1-186a-4120-be4a-61974d22fc8b,"[{'from': 'system', 'value': 'You are a functi...","[{'type': 'function', 'function': {'name': 'in...",IoT and Home Automation,Smart Home Setup,Set Up a Smart Home System,"[{'role': 'system', 'content': 'You are a func..."
3,NousResearch/hermes-function-calling-v1,func_calling,1,bd48c5c1-186a-4120-be4a-61974d22fc8b,"[{'from': 'system', 'value': 'You are a functi...","[{'type': 'function', 'function': {'name': 'in...",IoT and Home Automation,Smart Home Setup,Set Up a Smart Home System,"[{'role': 'system', 'content': 'You are a func..."
4,NousResearch/hermes-function-calling-v1,func_calling,2,b099e2c1-8459-4848-bd4f-0385e0d45d59,"[{'from': 'system', 'value': 'You are a functi...","[{'type': 'function', 'function': {'name': 'se...",IoT and Home Automation,Thermostat Control,Adjust Smart Thermostat Settings,"[{'role': 'system', 'content': 'You are a func..."


In [20]:
# duplicate rows N times 
print(df.shape)
df = pd.concat([df] * 10, ignore_index=True)
print(df.shape)
df.head()


(2759, 10)
(27590, 10)


,source_dataset,source_config,source_row_idx,id,conversations,tools,category,subcategory,task,messages
0,NousResearch/hermes-function-calling-v1,func_calling,0,82088305-310b-45cb-ac76-ab273503b5cd,"[{'from': 'system', 'value': 'You are a functi...","[{'type': 'function', 'function': {'name': 'ge...",IoT and Home Automation,Security Camera Management,View and Manage Security Camera Feeds,"[{'role': 'system', 'content': 'You are a func..."
1,NousResearch/hermes-function-calling-v1,func_calling,0,82088305-310b-45cb-ac76-ab273503b5cd,"[{'from': 'system', 'value': 'You are a functi...","[{'type': 'function', 'function': {'name': 'ge...",IoT and Home Automation,Security Camera Management,View and Manage Security Camera Feeds,"[{'role': 'system', 'content': 'You are a func..."
2,NousResearch/hermes-function-calling-v1,func_calling,1,bd48c5c1-186a-4120-be4a-61974d22fc8b,"[{'from': 'system', 'value': 'You are a functi...","[{'type': 'function', 'function': {'name': 'in...",IoT and Home Automation,Smart Home Setup,Set Up a Smart Home System,"[{'role': 'system', 'content': 'You are a func..."
3,NousResearch/hermes-function-calling-v1,func_calling,1,bd48c5c1-186a-4120-be4a-61974d22fc8b,"[{'from': 'system', 'value': 'You are a functi...","[{'type': 'function', 'function': {'name': 'in...",IoT and Home Automation,Smart Home Setup,Set Up a Smart Home System,"[{'role': 'system', 'content': 'You are a func..."
4,NousResearch/hermes-function-calling-v1,func_calling,2,b099e2c1-8459-4848-bd4f-0385e0d45d59,"[{'from': 'system', 'value': 'You are a functi...","[{'type': 'function', 'function': {'name': 'se...",IoT and Home Automation,Thermostat Control,Adjust Smart Thermostat Settings,"[{'role': 'system', 'content': 'You are a func..."


In [21]:
# add a new column to the dataset that will be used to determine the type of corruption

import numpy as np

AVAILABLE_CORRUPTION_TYPES = ['no_corruption', 'delete_content', 'corrupt_content', 'add_unnecessary_content', 'delete_tool_call', 'corrupt_tool_call_name', 'corrupt_tool_call_argument', 'add_extra_tool_call']
# for now give equal probability to each corruption type
df['corruption_type'] = np.random.choice(AVAILABLE_CORRUPTION_TYPES, size=df.shape[0])
df.head()

,source_dataset,source_config,source_row_idx,id,conversations,tools,category,subcategory,task,messages,corruption_type
0,NousResearch/hermes-function-calling-v1,func_calling,0,82088305-310b-45cb-ac76-ab273503b5cd,"[{'from': 'system', 'value': 'You are a functi...","[{'type': 'function', 'function': {'name': 'ge...",IoT and Home Automation,Security Camera Management,View and Manage Security Camera Feeds,"[{'role': 'system', 'content': 'You are a func...",corrupt_tool_call_name
1,NousResearch/hermes-function-calling-v1,func_calling,0,82088305-310b-45cb-ac76-ab273503b5cd,"[{'from': 'system', 'value': 'You are a functi...","[{'type': 'function', 'function': {'name': 'ge...",IoT and Home Automation,Security Camera Management,View and Manage Security Camera Feeds,"[{'role': 'system', 'content': 'You are a func...",add_extra_tool_call
2,NousResearch/hermes-function-calling-v1,func_calling,1,bd48c5c1-186a-4120-be4a-61974d22fc8b,"[{'from': 'system', 'value': 'You are a functi...","[{'type': 'function', 'function': {'name': 'in...",IoT and Home Automation,Smart Home Setup,Set Up a Smart Home System,"[{'role': 'system', 'content': 'You are a func...",corrupt_content
3,NousResearch/hermes-function-calling-v1,func_calling,1,bd48c5c1-186a-4120-be4a-61974d22fc8b,"[{'from': 'system', 'value': 'You are a functi...","[{'type': 'function', 'function': {'name': 'in...",IoT and Home Automation,Smart Home Setup,Set Up a Smart Home System,"[{'role': 'system', 'content': 'You are a func...",add_extra_tool_call
4,NousResearch/hermes-function-calling-v1,func_calling,2,b099e2c1-8459-4848-bd4f-0385e0d45d59,"[{'from': 'system', 'value': 'You are a functi...","[{'type': 'function', 'function': {'name': 'se...",IoT and Home Automation,Thermostat Control,Adjust Smart Thermostat Settings,"[{'role': 'system', 'content': 'You are a func...",add_extra_tool_call


In [74]:
# use llms for corrupting the messages
# corruption types for llms: corrupt_content, add_unnecessary_content, corrupt_tool_call_name, corrupt_tool_call_argument, add_extra_tool_call

CORRUPTION_PROMPT_BASE = """
You are generating a deliberately corrupted assistant draft for adapter-training.
You will receive:
- the original system prompt with tool definitions if any
- the original conversation history between the user and the assistant
- the gold assistant response that you will be corrupting

""".strip()

CORRUPTION_SYSTEM_PROMPTS = {
    "corrupt_content": "Corrupt the content of the assistant response.",
    "add_unnecessary_content": "Produce a draft with extra assistant text that should not be there.",
    "corrupt_tool_call_name": "Rewrite the given tool call with a wrong tool name.",
    "corrupt_tool_call_argument": "Rewrite the given tool call with wrong argument names or values or value types.",
    "add_extra_tool_call": "Produce an extra unnecessary tool call. ONLY ONE NEW TOOL CALL.",
}


CORRUPTION_PROMPT_RESPONSE_FORMAT = """
<corruption_output>
    <notes>[...]</notes>
    <content>if corrupted content</content>
    <tool_calls>if corrupted tool call</tool_calls>
</corruption_output>

Example:
<corruption_output>
    <notes>...</notes>
    <content>...</content>
    <tool_calls>{"arguments": {"arg1": "...", "arg2": "..."}, "name": "..."}</tool_calls>
</corruption_output>
"""


In [23]:
example

{'source_dataset': 'NousResearch/hermes-function-calling-v1',
 'source_config': 'func_calling',
 'source_row_idx': 1892,
 'id': '98c8fda0-ca02-4d3c-ac96-c5bd6bf6904a',
 'conversations': [{'from': 'system',
   'value': 'You are an expert structured information extraction AI model. You will be provided with documents to extract information from. You are also provided with the json schema to output extracted information in the function signatures within XML tags <tools></tools>. Don\'t make assumptions about what values to plug into json schema. \n<tools>\n[{"type": "function", "function": {"name": "ExpertQAExtractor", "description": "Extracts a list of questions that focus on summarizing a specific topic found in the document.", "parameters": {"type": "object", "properties": {"topic_summarization_questions": {"type": "array", "items": {"type": "string"}}}, "required": ["topic_summarization_questions"]}}}]\n</tools>\nFor each extraction function call return a json object with function nam

In [24]:
def get_conversation_system_prompt(messages, tools):
    system_prompt = messages[0]['content']
    system_prompt += f'\n\n<tools>\n{json.dumps(tools)}\n</tools>'
    return system_prompt

system_prompt = get_conversation_system_prompt(example['messages'], example['tools'])
print(system_prompt)


You are an expert structured information extraction AI model. You will be provided with documents to extract information from. You are also provided with the json schema to output extracted information in the function signatures within XML tags <tools></tools>. Don't make assumptions about what values to plug into json schema.

<tools>
[{"type": "function", "function": {"name": "ExpertQAExtractor", "description": "Extracts a list of questions that focus on summarizing a specific topic found in the document.", "parameters": {"type": "object", "properties": {"topic_summarization_questions": {"type": "array", "items": {"type": "string"}}}, "required": ["topic_summarization_questions"]}}}]
</tools>


In [25]:
def get_conversation_history(messages):
    CONVERSATION_HISTORY = []
    for msg in messages[1:-1]:  # ignore system prompt and final assistant response
        if msg['role'] == 'user' or msg['role'] == 'tool':
            CONVERSATION_HISTORY.append(f'{msg["role"]}: {msg["content"]}')
        if msg['role'] == 'assistant':
            s = f'{msg["role"]}: {msg["content"]}'
            if 'tool_calls' in msg:
                s += f'\n<tool_calls>\n{json.dumps(msg["tool_calls"])}\n</tool_calls>'
            CONVERSATION_HISTORY.append(s)

    return '\n---\n'.join(CONVERSATION_HISTORY)

CONVERSATION_HISTORY = get_conversation_history(messages)
print(CONVERSATION_HISTORY)


user: Can you help me extract queries from the following passage <passage> = - 3 x + y - z = 0 
C50+^ A three-digit number has two properties. The tens-digit and the ones-digit add up to 5. If the number is written with the digits in the reverse order, and then subtracted 
SSS S L E B e e z e r : A F i r s t C o u r s e i n L i n e a r A l g e b r a 16 
from the original number, the result is 792. Use a system of equations to find all of the three-digit numbers with these properties. 
C51+^ Find all of the six-digit numbers in which the first digit is one less than the second, the third digit is half the second, the fourth digit is three times the third and the last two digits form a number that equals the sum of the fourth and fifth. The sum of all the digits is 24. (From The MENSA Puzzle Calendar for January 9, 2006.) 
C52+^ Driving along, Terry notices that the last four digits on his car's odometer are palindromic. A mile later, the last five digits are palindromic. After driving a

In [26]:
assistant_content_to_corrupt = messages[-1].get('content', None)
assistant_tool_calls_to_corrupt = messages[-1].get('tool_calls', None)

print(assistant_content_to_corrupt)
print(assistant_tool_calls_to_corrupt)


<tool_call>\n{"arguments": {"queries": ['How is a matrix defined and what is its purpose in linear algebra?', 'What is the difference between a coefficient matrix and a vector of constants in a system of equations?', 'How is a system of equations represented in matrix notation?'], "name": "ExpertQAExtractor"}}\n</tool_call>
None


In [31]:
prompt = f"""system_prompt:
{system_prompt}

conversation:
{CONVERSATION_HISTORY}

gold_assistant_response:
{assistant_content_to_corrupt}
"""

corruption_prompt = f"""
{CORRUPTION_PROMPT_BASE}

{CORRUPTION_SYSTEM_PROMPTS['corrupt_tool_call_name']}
Response format:
{CORRUPTION_PROMPT_RESPONSE_FORMAT}
"""
print(corruption_prompt)
print('\n---\n')
print(prompt)


You are generating a deliberately corrupted assistant draft for adapter-training.
You will receive:
- the original system prompt with tool definitions if any
- the original conversation between the user and the assistant
- the gold assistant response

Rewrite the given tool call with a wrong tool name.
Response format:

<corruption_output>
    <notes>[...]</notes>
    <content>if corrupted content</content>
    <tool_calls>if corrupted tool call</tool_calls>
</corruption_output>

Example:
<corruption_output>
    <notes>...</notes>
    <content>...</content>
    <tool_calls>{"arguments": {"arg1": "...", "arg2": "..."}, "name": "..."}</tool_calls>
</corruption_output>



---

system_prompt:
You are an expert structured information extraction AI model. You will be provided with documents to extract information from. You are also provided with the json schema to output extracted information in the function signatures within XML tags <tools></tools>. Don't make assumptions about what value

In [32]:
response = client.chat.completions.create(
    model='glm-4.7-fp8',
    messages=[
        {"role": "system", "content": corruption_prompt},
        {"role": "user", "content": prompt}
    ],
    max_tokens=32672,
)
print('corruption response:')
print(response.choices[0].message.content)

corruption response:
<corruption_output>
    <notes>Changed the tool name from "ExpertQAExtractor" to "InvalidToolName".</notes>
    <content></content>
    <tool_calls>{"arguments": {"queries": ["How is a matrix defined and what is its purpose in linear algebra?", "What is the difference between a coefficient matrix and a vector of constants in a system of equations?", "How is a system of equations represented in matrix notation?"], "name": "InvalidToolName"}}</tool_calls>
</corruption_output>


In [29]:
response.choices[0].finish_reason

'stop'

In [84]:
content_ptrn = re.compile(r"<content>(.*?)</content>", flags=re.DOTALL)
tool_calls_ptrn = re.compile(r"<tool_calls>(.*?)</tool_calls>", flags=re.DOTALL)

def parse_llm_response(ptrn, llm_response):
    matches = ptrn.search(llm_response)
    if matches is None or len(matches.groups()) < 1: return None
    return matches.groups()[0].strip()

from openai import AsyncOpenAI
async_client = AsyncOpenAI(base_url='http://localhost:8000/v1', api_key='dummy')

async def llm_corruption(corruption_type, messages, tools, assistant_content_to_corrupt):
    system_prompt = get_conversation_system_prompt(messages, tools)
    CONVERSATION_HISTORY = get_conversation_history(messages)

    prompt = f"""system_prompt:
{system_prompt}

conversation:
{CONVERSATION_HISTORY}

gold_assistant_response:
{assistant_content_to_corrupt}

{CORRUPTION_SYSTEM_PROMPTS[corruption_type]}
    """.strip()

    corruption_prompt = f"""
{CORRUPTION_PROMPT_BASE}

{CORRUPTION_SYSTEM_PROMPTS[corruption_type]}
Response format:
{CORRUPTION_PROMPT_RESPONSE_FORMAT}
    """.strip()

    response = await async_client.chat.completions.create(
        model='glm-4.7-fp8',
        messages=[
            {"role": "system", "content": corruption_prompt},
            {"role": "user", "content": prompt}
        ],
        max_tokens=32672,
    )
    llm_response = response.choices[0].message.content
    content = parse_llm_response(content_ptrn, llm_response)
    tool_calls = parse_llm_response(tool_calls_ptrn, llm_response)

    return content, tool_calls


content, tool_calls = await llm_corruption('corrupt_tool_call_name', example['messages'], example['tools'], example['messages'][-1]['content'])
print(content)
print(tool_calls)

None
{"arguments": {"queries": ["How is a matrix defined and what is its purpose in linear algebra?", "What is the difference between a coefficient matrix and a vector of constants in a system of equations?", "How is a system of equations represented in matrix notation?"]}, "name": "ExtractionTool"}


In [52]:
s = """
<tool_call>
"""

tool_call_ptrn = re.compile(r"<tool_call>\n(.*)\n<\/tool_call>")
matches = re.findall(tool_call_ptrn, s)
len(matches)

0

In [51]:
matches[0]

'{"name": "get_camera_live_feed", "arguments": {"camera_id": "front_door", "stream_quality": "1080p"}}'

In [104]:
"""
Corruption requirements per type:
- delete_content: requires gold content to be non-empty.
- corrupt_content: requires gold content to be non-empty.
- add_unnecessary_content: can work with empty or non-empty gold content and/or tool calls.
- delete_tool_call: requires gold to have at least one tool call.
- corrupt_tool_call_name: requires gold to have at least one tool call.
- corrupt_tool_call_argument: requires gold to have at least one tool call.
- add_extra_tool_call: can work with empty or non-empty gold content and/or tool calls.
"""

tool_call_ptrn = re.compile(r"<tool_call>\n(.*)\n<\/tool_call>")

def extract_gold_content(assistant_response) -> str:
    content = tool_call_ptrn.sub('', assistant_response).strip()
    return content if content else None

def extract_gold_tool_calls(assistant_response) -> list[dict]:
    matches = re.findall(tool_call_ptrn, assistant_response)
    if len(matches) == 0: return None
    return matches

def tool_call_to_assistant_response(tool_calls: list[str]) -> str:
    return '\n'.join([f'<tool_call>\n{x}\n</tool_call>' for x in tool_calls])


async def corrupt(corruption_type, messages, tools) -> bool:
    assistant_response = messages[-1]['content'].strip()
    if corruption_type == 'no_corruption': return assistant_response

    gold_content = extract_gold_content(assistant_response)
    gold_tool_calls = extract_gold_tool_calls(assistant_response)

    if corruption_type == 'delete_content':
        return assistant_response.replace(gold_content, '') if gold_content else None

    if corruption_type == 'delete_tool_call':
        if gold_tool_calls is None: return None
        tool_calls = deepcopy(gold_tool_calls)
        tool_calls.pop(np.random.randint(0, len(tool_calls)))
        gold_tool_calls_str = tool_call_to_assistant_response(gold_tool_calls)
        tool_calls_str = tool_call_to_assistant_response(tool_calls)
        return assistant_response.replace(gold_tool_calls_str, tool_calls_str)

    if corruption_type == 'corrupt_content':
        if gold_content is None: return None
        content, _ = await llm_corruption('corrupt_content', messages, tools, gold_content)
        if content is None: return None
        return assistant_response.replace(gold_content, content)

    if corruption_type == 'corrupt_tool_call_name' or corruption_type == 'corrupt_tool_call_argument':
        if gold_tool_calls is None: return None
        tool_calls = deepcopy(gold_tool_calls)
        # randomly choose a tool call
        tool_call = tool_calls[np.random.randint(0, len(tool_calls))]
        _, corrupted_tool_call = await llm_corruption(corruption_type, messages, tools, tool_call)
        if corrupted_tool_call is None: return None
        return assistant_response.replace(tool_call, corrupted_tool_call)

    if corruption_type == 'add_unnecessary_content':
        if not gold_content: return None
        content, _ = await llm_corruption(corruption_type, messages, tools, gold_content)
        if content is None: return None
        return assistant_response.replace(gold_content, content)

    if corruption_type == 'add_extra_tool_call':
        _, corrupted_tool_call = await llm_corruption(corruption_type, messages, tools, gold_tool_calls)
        if corrupted_tool_call is None: return None
        corrupted_tool_call_str = tool_call_to_assistant_response([corrupted_tool_call])
        if gold_tool_calls:
            gold_tool_calls_str = tool_call_to_assistant_response(gold_tool_calls)
            return assistant_response.replace(gold_tool_calls_str, gold_tool_calls_str + f'\n{corrupted_tool_call_str}')
        return assistant_response + f'\n{corrupted_tool_call_str}'

    return assistant_response

import asyncio
from tqdm import tqdm
semaphore = asyncio.Semaphore(32)
async def limited_corrupt(corruption_type, messages, tools, pbar):
    async with semaphore:
        ret = await corrupt(corruption_type, messages, tools)
        pbar.update(1)
        return ret

# drop rows that don't meet the corruption requirements
corrupted_responses = []
tasks = []
sampled_df = df.sample(frac=1.0)
pbar = tqdm(total=len(sampled_df), ncols=80)
for i, row in sampled_df.iterrows():
    tasks.append(limited_corrupt(row['corruption_type'], row['messages'], row['tools'], pbar))

corrupted_responses = await asyncio.gather(*tasks, return_exceptions=True)
corrupted_responses = [None if isinstance(x, Exception) else x for x in corrupted_responses]
print(len(corrupted_responses))

100%|██████████████████████████████████▉| 27575/27590 [3:11:58<07:54, 31.65s/it]

27590


In [105]:
pbar.close()

100%|██████████████████████████████████▉| 27575/27590 [3:11:58<00:06,  2.39it/s]


In [106]:
print(sampled_df.shape)
sampled_df['corrupted_response'] = corrupted_responses
sampled_df['is_valid'] = sampled_df['corrupted_response'].notna()
sampled_df = sampled_df[sampled_df['is_valid']]
sampled_df = sampled_df.drop(columns=['is_valid'])
print(sampled_df.shape)
sampled_df.head()


(27590, 11)
(16728, 12)


,source_dataset,source_config,source_row_idx,id,conversations,tools,category,subcategory,task,messages,corruption_type,corrupted_response
18358,NousResearch/hermes-function-calling-v1,func_calling,977,2e583020-ea45-4496-aafe-a6d141ed1ad7,"[{'from': 'system', 'value': 'You are a functi...","[{'type': 'function', 'function': {'name': 'an...",Consumer Staples Software,Drug Retail Supply Chain Management,Streamline supply chain operations for drug re...,"[{'role': 'system', 'content': 'You are a func...",corrupt_tool_call_name,"<tool_call>\n{""arguments"": {""product_database""..."
19030,NousResearch/hermes-function-calling-v1,func_calling,1587,99abd602-3854-41e1-9fe7-31dcef70ad39,"[{'from': 'system', 'value': 'You are an exper...","[{'type': 'function', 'function': {'name': 'Ex...",Information Extraction,Json Schema,Structured json schema extaction with function...,"[{'role': 'system', 'content': 'You are an exp...",delete_content,
5026,NousResearch/hermes-function-calling-v1,func_calling,1360,beb07363-1986-4701-a4b7-27e04536f093,"[{'from': 'system', 'value': 'You are an exper...","[{'type': 'function', 'function': {'name': 'Ex...",Information Extraction,Json Schema,Structured json schema extaction with function...,"[{'role': 'system', 'content': 'You are an exp...",delete_content,
27447,NousResearch/hermes-function-calling-v1,func_calling,1738,28558c88-cfd6-431e-894a-3d7510b416ba,"[{'from': 'system', 'value': 'You are an exper...","[{'type': 'function', 'function': {'name': 'Ex...",Information Extraction,Json Schema,Structured json schema extaction with function...,"[{'role': 'system', 'content': 'You are an exp...",no_corruption,"<tool_call>\n{""arguments"": {""queries"": ['Can y..."
10531,NousResearch/hermes-function-calling-v1,func_calling,1347,248b82c3-e473-40e8-91ff-74d9b314cd1f,"[{'from': 'system', 'value': 'You are an exper...","[{'type': 'function', 'function': {'name': 'Ex...",Information Extraction,Json Schema,Structured json schema extaction with function...,"[{'role': 'system', 'content': 'You are an exp...",add_extra_tool_call,"<tool_call>\n{""arguments"": {""queries"": ['What ..."


In [108]:
for x in corrupted_responses:
    if isinstance(x, Exception):
        print(x)
        break

In [116]:
# drop duplicates
import hashlib
sampled_df['message_hashes'] = sampled_df['messages'].apply(lambda x: hashlib.sha256(json.dumps(x).encode()).hexdigest())
sampled_df = sampled_df.drop_duplicates(subset=['message_hashes', 'corrupted_response']).reset_index(drop=True)
sampled_df = sampled_df.drop(columns=['message_hashes'])
sampled_df.shape

(13538, 12)

In [117]:
sampled_df.to_pickle('./data/full_df_deduped.pkl')

In [122]:
# fix the corrupted response tool calls

sampled_df['corrupted_response'] = sampled_df['corrupted_response'].apply(lambda x: x.replace('\\n', '\n'))

In [123]:
sampled_df.to_pickle('./data/full_df_deduped_fixed_tool_calls.pkl')

In [124]:
sampled_df = pd.read_pickle('./data/full_df_deduped_fixed_tool_calls.pkl')
print(sampled_df.shape)
sampled_df.head()

(13538, 12)


,source_dataset,source_config,source_row_idx,id,conversations,tools,category,subcategory,task,messages,corruption_type,corrupted_response
0,NousResearch/hermes-function-calling-v1,func_calling,977,2e583020-ea45-4496-aafe-a6d141ed1ad7,"[{'from': 'system', 'value': 'You are a functi...","[{'type': 'function', 'function': {'name': 'an...",Consumer Staples Software,Drug Retail Supply Chain Management,Streamline supply chain operations for drug re...,"[{'role': 'system', 'content': 'You are a func...",corrupt_tool_call_name,"<tool_call>\n{""arguments"": {""product_database""..."
1,NousResearch/hermes-function-calling-v1,func_calling,1587,99abd602-3854-41e1-9fe7-31dcef70ad39,"[{'from': 'system', 'value': 'You are an exper...","[{'type': 'function', 'function': {'name': 'Ex...",Information Extraction,Json Schema,Structured json schema extaction with function...,"[{'role': 'system', 'content': 'You are an exp...",delete_content,
2,NousResearch/hermes-function-calling-v1,func_calling,1360,beb07363-1986-4701-a4b7-27e04536f093,"[{'from': 'system', 'value': 'You are an exper...","[{'type': 'function', 'function': {'name': 'Ex...",Information Extraction,Json Schema,Structured json schema extaction with function...,"[{'role': 'system', 'content': 'You are an exp...",delete_content,
3,NousResearch/hermes-function-calling-v1,func_calling,1738,28558c88-cfd6-431e-894a-3d7510b416ba,"[{'from': 'system', 'value': 'You are an exper...","[{'type': 'function', 'function': {'name': 'Ex...",Information Extraction,Json Schema,Structured json schema extaction with function...,"[{'role': 'system', 'content': 'You are an exp...",no_corruption,"<tool_call>\n{""arguments"": {""queries"": ['Can y..."
4,NousResearch/hermes-function-calling-v1,func_calling,1347,248b82c3-e473-40e8-91ff-74d9b314cd1f,"[{'from': 'system', 'value': 'You are an exper...","[{'type': 'function', 'function': {'name': 'Ex...",Information Extraction,Json Schema,Structured json schema extaction with function...,"[{'role': 'system', 'content': 'You are an exp...",add_extra_tool_call,"<tool_call>\n{""arguments"": {""queries"": ['What ..."
